In [1]:
import sys
if '/home/jovyan/src' not in sys.path:
    sys.path.append('/home/jovyan/src')

from spark_utils import crear_spark_session
from pyspark.sql import functions as F
from config import BRONZE_PATH, SILVER_PATH, GOLD_PATH, ANIOS, TIPOS_VEHICULO
import pandas as pd

spark = crear_spark_session()

resultados = []

for tipo in TIPOS_VEHICULO:
    for anio in ANIOS:
        path_bronze = BRONZE_PATH / tipo / str(anio)
        path_silver = SILVER_PATH / f"tipo_vehiculo={tipo}" / f"anio={anio}"

        if not path_bronze.exists():
            print(f"  [SKIP] No existe bronze: {tipo} {anio}")
            continue
        if not path_silver.exists():
            print(f"  [SKIP] No existe silver: {tipo} {anio}")
            continue

        n_bronze = spark.read.parquet(str(path_bronze)).count()
        n_silver = spark.read.parquet(str(path_silver)).count()

        resultados.append({
            "tipo_vehiculo": tipo,
            "anio": anio,
            "filas_bronze": n_bronze,
            "filas_silver": n_silver,
            "filas_eliminadas": n_bronze - n_silver,
            "pct_retenido": round(n_silver / n_bronze * 100, 2) if n_bronze > 0 else None
        })
        print(f"  {tipo} {anio}: bronze={n_bronze:,} -> silver={n_silver:,} ({resultados[-1]['pct_retenido']}%)")

df_auditoria = pd.DataFrame(resultados)
print("\n=== RESUMEN COMPLETO ===")
print(df_auditoria)

2026-07-17 05:29:11,820 [INFO] NumExpr defaulting to 4 threads.
2026-07-17 05:29:20,704 [INFO] SparkSession creada: 3.5.0


  yellow 2023: bronze=38,310,226 -> silver=37,073,817 (96.77%)
  yellow 2024: bronze=41,169,720 -> silver=39,601,734 (96.19%)
  yellow 2025: bronze=48,722,602 -> silver=44,056,579 (90.42%)
  green 2023: bronze=787,060 -> silver=736,157 (93.53%)
  green 2024: bronze=660,218 -> silver=615,492 (93.23%)
  green 2025: bronze=591,375 -> silver=551,377 (93.24%)
  fhv 2023: bronze=15,858,639 -> silver=3,396,389 (21.42%)
  fhv 2024: bronze=17,630,326 -> silver=3,427,405 (19.44%)
  fhv 2025: bronze=25,047,544 -> silver=4,394,998 (17.55%)
  fhvhv 2023: bronze=232,490,020 -> silver=231,304,577 (99.49%)
  fhvhv 2024: bronze=239,470,448 -> silver=239,281,228 (99.92%)
  fhvhv 2025: bronze=243,589,684 -> silver=243,083,536 (99.79%)

=== RESUMEN COMPLETO ===
   tipo_vehiculo  anio  filas_bronze  filas_silver  filas_eliminadas  \
0         yellow  2023      38310226      37073817           1236409   
1         yellow  2024      41169720      39601734           1567986   
2         yellow  2025      4872

In [2]:
from pyspark.sql import functions as F
import sys
if '/home/jovyan/src' not in sys.path:
    sys.path.append('/home/jovyan/src')
from config import BRONZE_PATH, UMBRALES, PAYMENT_TYPES_ERROR_NEGATIVO
from silver.cleaning_rules import _duracion_min_expr

anio_muestra = 2023  # ajusta si quieres verlo para otro año, o mételo en un loop
mes_muestra = 1

path = BRONZE_PATH / "yellow" / str(anio_muestra) / f"yellow_tripdata_{anio_muestra}-{mes_muestra:02d}.parquet"
df = spark.read.parquet(str(path))

# Ajusta nombres de columnas crudas de bronze si difieren tras el rename a silver
df = df.withColumnRenamed("tpep_pickup_datetime", "pickup_datetime") \
       .withColumnRenamed("tpep_dropoff_datetime", "dropoff_datetime")

dur = _duracion_min_expr()
df = df.withColumn("duracion_min", dur)

n_inicial = df.count()
print(f"Filas iniciales (bronze, {anio_muestra}-{mes_muestra:02d}): {n_inicial:,}")

resultados_cascada = []

# Paso 1: fecha corrupta
df_1 = df.filter(
    (F.year("pickup_datetime") == anio_muestra) & (F.month("pickup_datetime") == mes_muestra)
)
n1 = df_1.count()
resultados_cascada.append(("Fecha corrupta (año/mes no coincide)", n_inicial - n1, n1))

# Paso 2: fare negativo con payment_type inválido
df_2 = df_1.filter(
    ~((F.col("fare_amount") < 0) & (F.col("payment_type").isin(*PAYMENT_TYPES_ERROR_NEGATIVO)))
)
n2 = df_2.count()
resultados_cascada.append(("Fare negativo (payment_type inválido)", n1 - n2, n2))

# Paso 3: fare_amount fuera de rango [0, max]
df_3 = df_2.filter(
    (F.col("fare_amount") >= 0) & (F.col("fare_amount") <= UMBRALES["fare_amount_max"])
)
n3 = df_3.count()
resultados_cascada.append(("Fare fuera de rango [0, max]", n2 - n3, n3))

# Paso 4: GPS inválido (distancia=0 con duración alta)
df_4 = df_3.filter(
    ~((F.col("trip_distance") <= 0) & (F.col("duracion_min") > UMBRALES["gps_check_duracion_min"]))
)
n4 = df_4.count()
resultados_cascada.append(("GPS inválido (distancia≈0, duración alta)", n3 - n4, n4))

# Paso 5: distancia fuera de rango máximo
df_5 = df_4.filter(F.col("trip_distance") <= UMBRALES["trip_distance_max_taxi"])
n5 = df_5.count()
resultados_cascada.append(("Distancia > máximo", n4 - n5, n5))

# Paso 6: duración fuera de rango [min, max]
df_6 = df_5.filter(
    (F.col("duracion_min") >= UMBRALES["duracion_min_min"]) &
    (F.col("duracion_min") <= UMBRALES["duracion_min_max"])
)
n6 = df_6.count()
resultados_cascada.append(("Duración fuera de rango", n5 - n6, n6))

print(f"\nFilas finales (silver): {n6:,}\n")

import pandas as pd
df_cascada = pd.DataFrame(resultados_cascada, columns=["regla", "filas_eliminadas", "filas_restantes"])
df_cascada["pct_del_total_inicial"] = round(df_cascada["filas_eliminadas"] / n_inicial * 100, 3)
print(df_cascada)

Filas iniciales (bronze, 2023-01): 3,066,766

Filas finales (silver): 2,989,282

                                       regla  filas_eliminadas  \
0       Fecha corrupta (año/mes no coincide)                48   
1      Fare negativo (payment_type inválido)              5766   
2               Fare fuera de rango [0, max]             19496   
3  GPS inválido (distancia≈0, duración alta)             20366   
4                         Distancia > máximo                80   
5                    Duración fuera de rango             31728   

   filas_restantes  pct_del_total_inicial  
0          3066718                  0.002  
1          3060952                  0.188  
2          3041456                  0.636  
3          3021090                  0.664  
4          3021010                  0.003  
5          2989282                  1.035  


In [3]:
# Cuántas filas con fare negativo y payment_type válido (3,4) se están perdiendo por esta contradicción
n_perdidas_por_bug = df_2.filter(
    (F.col("fare_amount") < 0) & (~F.col("payment_type").isin(*PAYMENT_TYPES_ERROR_NEGATIVO))
).count()

print(f"Filas de devoluciones válidas (payment_type 3,4) eliminadas por error de lógica: {n_perdidas_por_bug:,}")

Filas de devoluciones válidas (payment_type 3,4) eliminadas por error de lógica: 19,283
